# BERT Fine-Tuning Across Multiple Prompts

This notebook fine-tunes `bert-base-uncased` as a prompt-based MLM classifier for multiple prompt templates from `bert_base_uncased.ipynb`.

For each prompt, it compares:
- base (not fine-tuned) MLM performance
- fine-tuned MLM performance

Each prompt has its own checkpoint, so retraining happens only when that prompt's training signature changes (or when `FORCE_RETRAIN=True`).
            


In [1]:
import hashlib
import json
import random
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import accuracy_score, classification_report, f1_score
from sklearn.model_selection import train_test_split
from torch.optim import AdamW
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm
from transformers import BertForMaskedLM, BertTokenizer, get_linear_schedule_with_warmup
            


In [2]:
# Reproducibility
RANDOM_STATE = 42


def seed_everything(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


seed_everything(RANDOM_STATE)

# Data + model config
DATA_PATH = "./data/preprocessed/steam_reviews_preprocessed.csv"
MODEL_NAME = "bert-base-uncased"
TEXT_COL = "review_info_text_cleaned"  # swap to review_info_text_cleaned if needed
DECIMAL_LABEL_COL = "label_funny_minmax"
WORDS_LABEL_COL = "label_votes_funny_categorical"

# Training config
MAX_LENGTH = 384
BATCH_SIZE = 16
EPOCHS = 3
LEARNING_RATE = 5e-5
WEIGHT_DECAY = 0.01
ZERO_CLASS_TO_NONZERO_RATIO = 1.0
MAX_ZERO_CLASS_SAMPLES = None
MAX_TOTAL_SAMPLES_PER_PROMPT = 10000  # hard cap after class-0 downsampling; None disables

# Note: Decimal prompts currently use minmax->rounded classes (0..9).
# Optional future alternative: keep class 0 for zero-vote reviews and
# create non-zero classes via log1p(review_votes_funny) quantile bins.

PROMPT_SPECS = [
    {
        "id": "decimal_0",
        "prompt": "On a scale from 0.0 to 1.0 this review is 0.[MASK] funny.",
        "label_words": ["0", "1", "2", "3", "4", "5", "6", "7", "8", "9"],
        "label_col": DECIMAL_LABEL_COL,
    },
    {
        "id": "decimal_1",
        "prompt": "On a scale from 0 to 1 this review is 0.[MASK] funny.",
        "label_words": ["0", "1", "2", "3", "4", "5", "6", "7", "8", "9"],
        "label_col": DECIMAL_LABEL_COL,
    },
    {
        "id": "words_0",
        "prompt": "Overall, the humor is [MASK].",
        "label_words": ["serious", "witty", "amusing", "hilarious", "hysterical"],
        "label_col": WORDS_LABEL_COL,
    },
    {
        "id": "words_1",
        "prompt": "Overall the humor of this review is [MASK].",
        "label_words": ["serious", "witty", "amusing", "hilarious", "hysterical"],
        "label_col": WORDS_LABEL_COL,
    },
    {
        "id": "words_2",
        "prompt": "The humor in this review is [MASK].",
        "label_words": ["serious", "witty", "amusing", "hilarious", "hysterical"],
        "label_col": WORDS_LABEL_COL,
    },
]

PROMPT_IDS_TO_RUN = [spec["id"] for spec in PROMPT_SPECS]

# Output + checkpoint config (separated by input text source)
def make_safe_name(value: str) -> str:
    return "".join(ch if ch.isalnum() else "_" for ch in value).strip("_").lower()


TEXT_VARIANT_NAME = {
    "review_text_cleaned": "review_text_only",
    "review_info_text_cleaned": "review_text_app_info",
}.get(TEXT_COL, make_safe_name(TEXT_COL))

RUN_ROOT = Path("./models/bert_finetuned_mlm") / TEXT_VARIANT_NAME
CHECKPOINT_ROOT = RUN_ROOT / "checkpoints"
ARTIFACTS_ROOT = RUN_ROOT / "artifacts"
CHECKPOINT_META_FILE = "training_metadata.json"
FORCE_RETRAIN = False
SAVE_FINETUNED_MODEL = True
SAVE_PROMPT_ARTIFACTS = True

print(f"Run root: {RUN_ROOT}")
print(f"Checkpoint root: {CHECKPOINT_ROOT}")
print(f"Artifacts root: {ARTIFACTS_ROOT}")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")


Run root: models\bert_finetuned_mlm\review_text_app_info
Checkpoint root: models\bert_finetuned_mlm\review_text_app_info\checkpoints
Artifacts root: models\bert_finetuned_mlm\review_text_app_info\artifacts
Using device: cuda


In [3]:
tokenizer = BertTokenizer.from_pretrained(MODEL_NAME)


def validate_prompt_spec(prompt_spec: dict) -> list[int]:
    words = prompt_spec["label_words"]
    if len(words) < 2:
        raise ValueError(
            f"Prompt '{prompt_spec['id']}' must have at least 2 label words."
        )

    if "label_col" not in prompt_spec:
        raise ValueError(f"Prompt '{prompt_spec['id']}' is missing label_col.")

    token_ids = []
    for word in words:
        toks = tokenizer.tokenize(word)
        if len(toks) != 1:
            raise ValueError(
                f"Prompt '{prompt_spec['id']}' label word '{word}' is not single-token: {toks}"
            )
        token_ids.append(tokenizer.convert_tokens_to_ids(toks[0]))
    return token_ids


prompt_id_to_spec = {spec["id"]: spec for spec in PROMPT_SPECS}
for pid in PROMPT_IDS_TO_RUN:
    spec = prompt_id_to_spec[pid]
    tok_ids = validate_prompt_spec(spec)
    print(
        f"{pid}: label_col={spec['label_col']} classes={len(spec['label_words'])} "
        f"prompt_len={len(tokenizer.tokenize(spec['prompt']))} token_ids={tok_ids}"
    )



decimal_0: label_col=label_funny_minmax classes=10 prompt_len=19 token_ids=[1014, 1015, 1016, 1017, 1018, 1019, 1020, 1021, 1022, 1023]
decimal_1: label_col=label_funny_minmax classes=10 prompt_len=15 token_ids=[1014, 1015, 1016, 1017, 1018, 1019, 1020, 1021, 1022, 1023]
words_0: label_col=label_votes_funny_categorical classes=5 prompt_len=7 token_ids=[3809, 25591, 19142, 26316, 25614]
words_1: label_col=label_votes_funny_categorical classes=5 prompt_len=9 token_ids=[3809, 25591, 19142, 26316, 25614]
words_2: label_col=label_votes_funny_categorical classes=5 prompt_len=8 token_ids=[3809, 25591, 19142, 26316, 25614]


In [4]:
# Load dataset once; per-prompt filtering/splitting happens later.
raw_df = pd.read_csv(DATA_PATH)

required_cols = {TEXT_COL, DECIMAL_LABEL_COL, WORDS_LABEL_COL, "review_id"}
missing_cols = sorted(required_cols - set(raw_df.columns))
if missing_cols:
    raise ValueError(f"Missing required columns: {missing_cols}")

print(f"Rows loaded: {len(raw_df):,}")
print("Non-null labels:")
print(raw_df[[DECIMAL_LABEL_COL, WORDS_LABEL_COL]].notna().sum())



Rows loaded: 722,790
Non-null labels:
label_funny_minmax               722790
label_votes_funny_categorical    722790
dtype: int64


In [5]:
def hash_review_ids(frame: pd.DataFrame) -> str:
    h = hashlib.sha256()
    for rid in frame["review_id"].astype(str).tolist():
        h.update(rid.encode("utf-8"))
        h.update(b"|")
    return h.hexdigest()[:16]


def build_prompt_splits(prompt_spec: dict):
    label_col = prompt_spec["label_col"]
    num_classes = len(prompt_spec["label_words"])

    class_label_col = "_tmp_class_label"

    df = raw_df.dropna(subset=[TEXT_COL, label_col]).copy()

    if label_col == DECIMAL_LABEL_COL:
        # Decimal labels are stored as 0.0..0.9, convert to class ids 0..9.
        df[class_label_col] = np.rint(df[label_col].astype(float) * 10).astype(int)
    else:
        df[class_label_col] = df[label_col].astype(int)

    df = df[df[class_label_col].between(0, num_classes - 1)].copy()

    class_counts = df[class_label_col].value_counts().sort_index()
    if class_counts.empty:
        raise ValueError(
            f"Prompt '{prompt_spec['id']}' has no usable rows for label_col '{label_col}'."
        )

    nonzero_counts = class_counts[class_counts.index != 0]
    if nonzero_counts.empty:
        raise ValueError(
            f"Prompt '{prompt_spec['id']}' has no non-zero classes after filtering."
        )

    min_nonzero_count = int(nonzero_counts.min())
    if min_nonzero_count < 10:
        raise ValueError(
            f"Prompt '{prompt_spec['id']}' has too few samples in non-zero classes: min={min_nonzero_count}."
        )

    zero_count_before = int(class_counts.get(0, 0))
    nonzero_total = int(nonzero_counts.sum())

    zero_target = int(round(nonzero_total * ZERO_CLASS_TO_NONZERO_RATIO))
    zero_target = max(1, zero_target)
    if MAX_ZERO_CLASS_SAMPLES is not None:
        zero_target = min(zero_target, int(MAX_ZERO_CLASS_SAMPLES))
    zero_target = min(zero_target, zero_count_before)

    if zero_count_before > 0:
        zero_df = df[df[class_label_col] == 0]
        if zero_target < zero_count_before:
            zero_df = zero_df.sample(n=zero_target, random_state=RANDOM_STATE)
    else:
        zero_df = df.iloc[0:0].copy()

    nonzero_df = df[df[class_label_col] != 0]

    sampled_df = (
        pd.concat([zero_df, nonzero_df], ignore_index=True)
        .sample(frac=1.0, random_state=RANDOM_STATE)
        .reset_index(drop=True)
    )

    total_before_cap = int(len(sampled_df))
    if MAX_TOTAL_SAMPLES_PER_PROMPT is not None:
        cap = int(MAX_TOTAL_SAMPLES_PER_PROMPT)
        if cap < num_classes * 2:
            raise ValueError(
                f"MAX_TOTAL_SAMPLES_PER_PROMPT={cap} too small for {num_classes} classes."
            )
        if total_before_cap > cap:
            sampled_df, _ = train_test_split(
                sampled_df,
                train_size=cap,
                random_state=RANDOM_STATE,
                stratify=sampled_df[class_label_col],
            )

    sampled_counts = sampled_df[class_label_col].value_counts().sort_index()
    total_after_cap = int(len(sampled_df))

    train_df, temp_df = train_test_split(
        sampled_df,
        test_size=0.2,
        random_state=RANDOM_STATE,
        stratify=sampled_df[class_label_col],
    )
    val_df, test_df = train_test_split(
        temp_df,
        test_size=0.5,
        random_state=RANDOM_STATE,
        stratify=temp_df[class_label_col],
    )

    split_signature = {
        "label_col": label_col,
        "num_classes": num_classes,
        "zero_count_before": zero_count_before,
        "zero_count_after": int(sampled_counts.get(0, 0)),
        "nonzero_total": nonzero_total,
        "min_nonzero_count": min_nonzero_count,
        "sampled_class_counts": {
            str(k): int(v) for k, v in sampled_counts.items()
        },
        "total_before_cap": total_before_cap,
        "total_after_cap": total_after_cap,
        "max_total_samples_per_prompt": MAX_TOTAL_SAMPLES_PER_PROMPT,
        "train_size": int(len(train_df)),
        "val_size": int(len(val_df)),
        "test_size": int(len(test_df)),
        "train_hash": hash_review_ids(train_df),
        "val_hash": hash_review_ids(val_df),
        "test_hash": hash_review_ids(test_df),
    }

    print(
        f"Prompt {prompt_spec['id']} | label_col={label_col} | classes={num_classes} | "
        f"class0 {zero_count_before:,}->{int(sampled_counts.get(0, 0)):,} | "
        f"nonzero_total={nonzero_total:,} | "
        f"total {total_before_cap:,}->{total_after_cap:,} | "
        f"train/val/test={len(train_df):,}/{len(val_df):,}/{len(test_df):,}"
    )

    return train_df, val_df, test_df, split_signature, class_label_col


In [6]:
class HumorMaskDataset(Dataset):
    def __init__(
        self,
        frame: pd.DataFrame,
        tokenizer: BertTokenizer,
        text_col: str,
        label_col: str,
        prompt: str,
        label_word_ids: list[int],
        max_length: int = 256,
    ):
        self.frame = frame.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.text_col = text_col
        self.label_col = label_col
        self.prompt = prompt
        self.label_word_ids = label_word_ids
        self.max_length = max_length

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, idx: int):
        row = self.frame.iloc[idx]
        text = str(row[self.text_col])
        class_label = int(row[self.label_col])

        enc = self.tokenizer(
            text,
            self.prompt,
            truncation="only_first",
            max_length=self.max_length,
            padding="max_length",
            return_tensors="pt",
        )

        input_ids = enc["input_ids"].squeeze(0)
        attention_mask = enc["attention_mask"].squeeze(0)

        labels = torch.full_like(input_ids, fill_value=-100)
        mask_positions = (input_ids == self.tokenizer.mask_token_id).nonzero(as_tuple=True)[0]
        if len(mask_positions) != 1:
            raise ValueError(f"Expected exactly one [MASK], got {len(mask_positions)}")

        labels[mask_positions.item()] = self.label_word_ids[class_label]

        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "labels": labels,
            "class_label": torch.tensor(class_label, dtype=torch.long),
            "review_id": str(row.get("review_id", idx)),
        }
            


In [7]:
def extract_label_word_probs(
    logits: torch.Tensor,
    input_ids: torch.Tensor,
    label_word_ids: list[int],
) -> torch.Tensor:
    label_word_ids_tensor = torch.tensor(label_word_ids, dtype=torch.long, device=logits.device)
    mask_positions = (input_ids == tokenizer.mask_token_id).nonzero(as_tuple=False)
    if mask_positions.shape[0] != input_ids.size(0):
        raise ValueError("Each sequence must contain exactly one [MASK] token.")

    batch_indices = torch.arange(input_ids.size(0), device=input_ids.device)
    mask_token_indices = mask_positions[:, 1]
    masked_logits = logits[batch_indices, mask_token_indices, :]
    label_logits = masked_logits.index_select(dim=1, index=label_word_ids_tensor)
    return torch.softmax(label_logits, dim=1)


@torch.no_grad()
def evaluate_model(
    model: BertForMaskedLM,
    loader: DataLoader,
    label_words: list[str],
    label_word_ids: list[int],
):
    model.eval()
    rows = []
    losses = []

    for batch in tqdm(loader, desc="Evaluating", leave=False):
        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        labels = batch["labels"].to(DEVICE)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        losses.append(float(outputs.loss.item()))

        probs = extract_label_word_probs(outputs.logits, input_ids, label_word_ids)
        preds = probs.argmax(dim=1).detach().cpu().numpy()
        true = batch["class_label"].detach().cpu().numpy()
        probs_np = probs.detach().cpu().numpy()
        review_ids = batch["review_id"]

        for i in range(len(true)):
            row = {
                "review_id": review_ids[i],
                "true_label": int(true[i]),
                "pred_label": int(preds[i]),
            }
            for cls_idx, word in enumerate(label_words):
                row[f"prob_{word}"] = float(probs_np[i, cls_idx])
            row["true_word"] = label_words[row["true_label"]]
            row["pred_word"] = label_words[row["pred_label"]]
            row["true_prob"] = row[f"prob_{row['true_word']}"]
            rows.append(row)

    pred_df = pd.DataFrame(rows)
    metrics = {
        "loss": float(np.mean(losses)),
        "accuracy": float(accuracy_score(pred_df["true_label"], pred_df["pred_label"])),
        "macro_f1": float(
            f1_score(pred_df["true_label"], pred_df["pred_label"], average="macro")
        ),
        "mean_true_prob": float(pred_df["true_prob"].mean()),
    }

    report = classification_report(
        pred_df["true_label"],
        pred_df["pred_label"],
        target_names=[f"{i}:{w}" for i, w in enumerate(label_words)],
        digits=4,
        zero_division=0,
    )

    conf = pd.crosstab(
        pred_df["true_label"],
        pred_df["pred_label"],
        rownames=["true"],
        colnames=["pred"],
    )
    return metrics, pred_df, report, conf
            


In [8]:
def train_one_epoch(model: BertForMaskedLM, loader: DataLoader, optimizer, scheduler=None):
    model.train()
    batch_losses = []

    for batch in tqdm(loader, desc="Training", leave=False):
        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        labels = batch["labels"].to(DEVICE)

        optimizer.zero_grad(set_to_none=True)
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        if scheduler is not None:
            scheduler.step()

        batch_losses.append(float(loss.item()))

    return float(np.mean(batch_losses))
            


In [9]:
def build_training_signature(prompt_spec: dict, split_signature: dict) -> dict:
    return {
        "model_name": MODEL_NAME,
        "prompt_id": prompt_spec["id"],
        "prompt": prompt_spec["prompt"],
        "label_words": prompt_spec["label_words"],
        "text_col": TEXT_COL,
        "label_col": prompt_spec["label_col"],
        "max_length": MAX_LENGTH,
        "batch_size": BATCH_SIZE,
        "epochs": EPOCHS,
        "learning_rate": LEARNING_RATE,
        "weight_decay": WEIGHT_DECAY,
        "zero_class_to_nonzero_ratio": ZERO_CLASS_TO_NONZERO_RATIO,
        "max_zero_class_samples": MAX_ZERO_CLASS_SAMPLES,
        "max_total_samples_per_prompt": MAX_TOTAL_SAMPLES_PER_PROMPT,
        "random_state": RANDOM_STATE,
        "split_signature": split_signature,
    }


def run_prompt_experiment(prompt_spec: dict):
    prompt_id = prompt_spec["id"]
    prompt = prompt_spec["prompt"]
    label_words = prompt_spec["label_words"]
    label_col = prompt_spec["label_col"]
    label_word_ids = validate_prompt_spec(prompt_spec)

    train_df, val_df, test_df, split_signature, class_label_col = build_prompt_splits(prompt_spec)

    train_ds = HumorMaskDataset(
        train_df,
        tokenizer,
        TEXT_COL,
        class_label_col,
        prompt,
        label_word_ids,
        MAX_LENGTH,
    )
    val_ds = HumorMaskDataset(
        val_df,
        tokenizer,
        TEXT_COL,
        class_label_col,
        prompt,
        label_word_ids,
        MAX_LENGTH,
    )
    test_ds = HumorMaskDataset(
        test_df,
        tokenizer,
        TEXT_COL,
        class_label_col,
        prompt,
        label_word_ids,
        MAX_LENGTH,
    )

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
    test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

    print(f"\n=== Prompt: {prompt_id} ===")
    print(prompt)

    base_model = BertForMaskedLM.from_pretrained(MODEL_NAME).to(DEVICE)
    base_metrics, base_pred_df, base_report, base_conf = evaluate_model(
        base_model, test_loader, label_words, label_word_ids
    )
    del base_model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    signature = build_training_signature(prompt_spec, split_signature)
    signature_hash = hashlib.sha256(
        json.dumps(signature, sort_keys=True).encode("utf-8")
    ).hexdigest()[:16]

    checkpoint_dir = CHECKPOINT_ROOT / prompt_id
    meta_path = checkpoint_dir / CHECKPOINT_META_FILE

    history = []
    loaded_from_checkpoint = False

    if not FORCE_RETRAIN and checkpoint_dir.exists() and meta_path.exists():
        saved_meta = json.loads(meta_path.read_text(encoding="utf-8"))
        if saved_meta.get("training_signature") == signature:
            finetuned_model = BertForMaskedLM.from_pretrained(str(checkpoint_dir)).to(DEVICE)
            history = saved_meta.get("history", [])
            loaded_from_checkpoint = True
            print(f"Loaded checkpoint: {checkpoint_dir} (signature={signature_hash})")
        else:
            print("Checkpoint signature mismatch; retraining.")

    if FORCE_RETRAIN:
        print("FORCE_RETRAIN=True; retraining.")

    if not loaded_from_checkpoint:
        finetuned_model = BertForMaskedLM.from_pretrained(MODEL_NAME).to(DEVICE)
        optimizer = AdamW(
            finetuned_model.parameters(),
            lr=LEARNING_RATE,
            weight_decay=WEIGHT_DECAY,
        )

        total_steps = EPOCHS * len(train_loader)
        scheduler = get_linear_schedule_with_warmup(
            optimizer,
            num_warmup_steps=max(1, int(0.1 * total_steps)),
            num_training_steps=total_steps,
        )

        for epoch in range(1, EPOCHS + 1):
            train_loss = train_one_epoch(finetuned_model, train_loader, optimizer, scheduler)
            val_metrics, _, _, _ = evaluate_model(
                finetuned_model, val_loader, label_words, label_word_ids
            )

            row = {
                "epoch": epoch,
                "train_loss": train_loss,
                "val_loss": val_metrics["loss"],
                "val_accuracy": val_metrics["accuracy"],
                "val_macro_f1": val_metrics["macro_f1"],
                "val_mean_true_prob": val_metrics["mean_true_prob"],
            }
            history.append(row)
            print(pd.Series(row).round(4))

        if SAVE_FINETUNED_MODEL:
            checkpoint_dir.mkdir(parents=True, exist_ok=True)
            finetuned_model.save_pretrained(str(checkpoint_dir))
            tokenizer.save_pretrained(str(checkpoint_dir))
            save_meta = {
                "saved_at_utc": datetime.now(timezone.utc).isoformat(),
                "base_model": MODEL_NAME,
                "prompt_id": prompt_id,
                "training_signature_hash": signature_hash,
                "training_signature": signature,
                "history": history,
            }
            meta_path.write_text(json.dumps(save_meta, indent=2), encoding="utf-8")
            print(f"Saved checkpoint: {checkpoint_dir}")

    finetuned_metrics, finetuned_pred_df, finetuned_report, finetuned_conf = evaluate_model(
        finetuned_model, test_loader, label_words, label_word_ids
    )
    del finetuned_model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    summary = {
        "prompt_id": prompt_id,
        "prompt": prompt,
        "label_col": label_col,
        "text_variant": TEXT_VARIANT_NAME,
        "label_words": ", ".join(label_words),
        "loaded_from_checkpoint": loaded_from_checkpoint,
        "base_loss": base_metrics["loss"],
        "base_accuracy": base_metrics["accuracy"],
        "base_macro_f1": base_metrics["macro_f1"],
        "base_mean_true_prob": base_metrics["mean_true_prob"],
        "finetuned_loss": finetuned_metrics["loss"],
        "finetuned_accuracy": finetuned_metrics["accuracy"],
        "finetuned_macro_f1": finetuned_metrics["macro_f1"],
        "finetuned_mean_true_prob": finetuned_metrics["mean_true_prob"],
        "delta_accuracy": finetuned_metrics["accuracy"] - base_metrics["accuracy"],
        "delta_macro_f1": finetuned_metrics["macro_f1"] - base_metrics["macro_f1"],
        "delta_mean_true_prob": (
            finetuned_metrics["mean_true_prob"] - base_metrics["mean_true_prob"]
        ),
    }

    return {
        "summary": summary,
        "history": pd.DataFrame(history),
        "base_pred_df": base_pred_df,
        "finetuned_pred_df": finetuned_pred_df,
        "base_report": base_report,
        "finetuned_report": finetuned_report,
        "base_conf": base_conf,
        "finetuned_conf": finetuned_conf,
    }


In [10]:
# Run all selected prompts and compare base vs fine-tuned
experiment_results = {}
summary_rows = []

for prompt_id in PROMPT_IDS_TO_RUN:
    if prompt_id not in prompt_id_to_spec:
        raise ValueError(f"Unknown prompt_id: {prompt_id}")

    result = run_prompt_experiment(prompt_id_to_spec[prompt_id])
    experiment_results[prompt_id] = result
    summary_rows.append(result["summary"])

summary_df = pd.DataFrame(summary_rows).sort_values("delta_macro_f1", ascending=False)

print("\nPer-prompt comparison (test set):")
display(
    summary_df[
        [
            "prompt_id",
            "loaded_from_checkpoint",
            "base_accuracy",
            "finetuned_accuracy",
            "delta_accuracy",
            "base_macro_f1",
            "finetuned_macro_f1",
            "delta_macro_f1",
            "base_mean_true_prob",
            "finetuned_mean_true_prob",
            "delta_mean_true_prob",
        ]
    ].round(4)
)
            


Prompt decimal_0 | label_col=label_funny_minmax | classes=10 | class0 720,391->2,399 | nonzero_total=2,399 | total 4,798->4,798 | train/val/test=3,838/480/480

=== Prompt: decimal_0 ===
On a scale from 0.0 to 1.0 this review is 0.[MASK] funny.


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

BertForMaskedLM LOAD REPORT from: bert-base-uncased
Key                         | Status     |  | 
----------------------------+------------+--+-
bert.pooler.dense.bias      | UNEXPECTED |  | 
bert.pooler.dense.weight    | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 
cls.seq_relationship.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Evaluating:   0%|          | 0/30 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved checkpoint: models\bert_finetuned_mlm\review_text_app_info\checkpoints\words_2


Evaluating:   0%|          | 0/63 [00:00<?, ?it/s]


Per-prompt comparison (test set):


,prompt_id,loaded_from_checkpoint,base_accuracy,finetuned_accuracy,delta_accuracy,base_macro_f1,finetuned_macro_f1,delta_macro_f1,base_mean_true_prob,finetuned_mean_true_prob,delta_mean_true_prob
3,words_1,False,0.0980,0.5680,0.4700,0.0396,0.2851,0.2454,0.1513,0.4962,0.3449
4,words_2,False,0.0850,0.5630,0.4780,0.0513,0.2707,0.2194,0.1428,0.4937,0.3509
2,words_0,False,0.1030,0.5670,0.4640,0.0803,0.2868,0.2065,0.1630,0.4967,0.3337
1,decimal_1,False,0.0229,0.5958,0.5729,0.0060,0.1710,0.1651,0.0443,0.5206,0.4763
0,decimal_0,False,0.0292,0.6062,0.5771,0.0077,0.1609,0.1532,0.1136,0.5163,0.4027


In [11]:
# Inspect detailed reports for one prompt
INSPECT_PROMPT_ID = PROMPT_IDS_TO_RUN[0]
inspect_result = experiment_results[INSPECT_PROMPT_ID]

print(f"Prompt: {INSPECT_PROMPT_ID}")
print("\nBASE classification report")
print(inspect_result["base_report"])
print("\nFINE-TUNED classification report")
print(inspect_result["finetuned_report"])

print("\nBASE confusion matrix")
display(inspect_result["base_conf"])
print("\nFINE-TUNED confusion matrix")
display(inspect_result["finetuned_conf"])
            


Prompt: decimal_0

BASE classification report
              precision    recall  f1-score   support

         0:0     0.2941    0.0208    0.0389       240
         1:1     0.0000    0.0000    0.0000       106
         2:2     0.0000    0.0000    0.0000        38
         3:3     0.0000    0.0000    0.0000        20
         4:4     0.0000    0.0000    0.0000        14
         5:5     0.0194    0.9000    0.0381        10
         6:6     0.0000    0.0000    0.0000         9
         7:7     0.0000    0.0000    0.0000         6
         8:8     0.0000    0.0000    0.0000         4
         9:9     0.0000    0.0000    0.0000        33

    accuracy                         0.0292       480
   macro avg     0.0314    0.0921    0.0077       480
weighted avg     0.1475    0.0292    0.0202       480


FINE-TUNED classification report
              precision    recall  f1-score   support

         0:0     0.7803    0.8583    0.8175       240
         1:1     0.4053    0.7264    0.5203       10

pred,0,5
true,,
0,5,235
1,4,102
2,2,36
3,2,18
4,0,14
5,1,9
6,1,8
7,0,6
8,1,3



FINE-TUNED confusion matrix


pred,0,1,9
true,,,
0,206,33,1
1,25,77,4
2,10,24,4
3,9,10,1
4,3,9,2
5,2,6,2
6,1,5,3
7,2,4,0
8,1,2,1


In [12]:
# Save per-prompt artifacts for later analysis
ARTIFACTS_ROOT.mkdir(parents=True, exist_ok=True)

summary_out_path = ARTIFACTS_ROOT / "prompt_comparison_summary.csv"
summary_df.to_csv(summary_out_path, index=False)
print(f"Saved: {summary_out_path}")

run_config_out_path = ARTIFACTS_ROOT / "run_config.json"
run_config_out_path.write_text(
    json.dumps(
        {
            "text_col": TEXT_COL,
            "text_variant_name": TEXT_VARIANT_NAME,
            "model_name": MODEL_NAME,
            "max_length": MAX_LENGTH,
            "batch_size": BATCH_SIZE,
            "epochs": EPOCHS,
            "learning_rate": LEARNING_RATE,
            "weight_decay": WEIGHT_DECAY,
            "zero_class_to_nonzero_ratio": ZERO_CLASS_TO_NONZERO_RATIO,
            "max_zero_class_samples": MAX_ZERO_CLASS_SAMPLES,
            "max_total_samples_per_prompt": MAX_TOTAL_SAMPLES_PER_PROMPT,
            "prompt_ids": PROMPT_IDS_TO_RUN,
        },
        indent=2,
    ),
    encoding="utf-8",
)
print(f"Saved: {run_config_out_path}")

if SAVE_PROMPT_ARTIFACTS:
    for prompt_id, result in experiment_results.items():
        prompt_dir = ARTIFACTS_ROOT / prompt_id
        prompt_dir.mkdir(parents=True, exist_ok=True)

        result["history"].to_csv(prompt_dir / "history.csv", index=False)
        result["base_pred_df"].to_csv(prompt_dir / "base_predictions.csv", index=False)
        result["finetuned_pred_df"].to_csv(
            prompt_dir / "finetuned_predictions.csv", index=False
        )
        result["base_conf"].to_csv(prompt_dir / "base_confusion_matrix.csv")
        result["finetuned_conf"].to_csv(prompt_dir / "finetuned_confusion_matrix.csv")

        (prompt_dir / "base_classification_report.txt").write_text(
            result["base_report"], encoding="utf-8"
        )
        (prompt_dir / "finetuned_classification_report.txt").write_text(
            result["finetuned_report"], encoding="utf-8"
        )

        summary_row = summary_df.loc[summary_df["prompt_id"] == prompt_id].iloc[0].to_dict()
        clean_summary_row = {
            k: (v.item() if hasattr(v, "item") else v) for k, v in summary_row.items()
        }
        (prompt_dir / "summary.json").write_text(
            json.dumps(clean_summary_row, indent=2), encoding="utf-8"
        )

    print(f"Saved detailed prompt artifacts under: {ARTIFACTS_ROOT}")


Saved: models\bert_finetuned_mlm\review_text_app_info\artifacts\prompt_comparison_summary.csv
Saved: models\bert_finetuned_mlm\review_text_app_info\artifacts\run_config.json
Saved detailed prompt artifacts under: models\bert_finetuned_mlm\review_text_app_info\artifacts


# Clean up to free up memory space

In [ ]:
# del (
#     experiment_results,
#     summary_rows,
#     summary_df,
#     run_config_out_path,
#     summary_out_path,
#     inspect_result,
#     INSPECT_PROMPT_ID,
#     PROMPT_IDS_TO_RUN,
#     prompt_id_to_spec,
#     PROMPT_SPECS,
#     MODEL_NAME,
#     TEXT_COL,
#     DECIMAL_LABEL_COL,
#     WORDS_LABEL_COL,
#     MAX_LENGTH,
#     BATCH_SIZE,
#     EPOCHS,
#     LEARNING_RATE,
#     WEIGHT_DECAY,
#     ZERO_CLASS_TO_NONZERO_RATIO,
#     MAX_ZERO_CLASS_SAMPLES,
#     MAX_TOTAL_SAMPLES_PER_PROMPT,
#     DEVICE,
#     tokenizer,
#     validate_prompt_spec,
#     build_prompt_splits,
#     HumorMaskDataset,
#     extract_label_word_probs,
#     evaluate_model,
#     train_one_epoch,
#     build_training_signature,
#     run_prompt_experiment,
# )

# if torch.cuda.is_available():
#     torch.cuda.empty_cache()